# Выбор следующего токена

Дополнение к § 3.4. Маленькие учебные распределения позволяют увидеть разницу между жадным выбором и сэмплированием без скачивания весов.

## Один шаг
Проверьте сумму долей и лидера.

In [ ]:
import numpy as np
words = np.array(["фигурку","задачу","вчера"])
prob = np.array([0.6,0.3,0.1])
assert np.isclose(prob.sum(), 1)
print("Жадно:", words[np.argmax(prob)])

## Повторяем случайный выбор
Измените seed и сравните десять выборов. Короткая серия не обязана точно повторять доли.

In [ ]:
rng = np.random.default_rng(7)
chosen = rng.choice(words, size=10, p=prob)
print(chosen.tolist())

## Меняем распределение
Предположите, какое слово станет лидером после изменения логитов, затем вычислите softmax.

In [ ]:
logits = np.array([2.0,1.0,0.0])
logits[1] += 1.5
shift = logits - logits.max()
p2 = np.exp(shift)/np.exp(shift).sum()
print(list(zip(words,p2.round(3))))

## Опыт с двумя началами

Запишите предположение, какое слово или тема появится после каждого начала. Установите зависимости статьи, включите run_model и сравните жадную генерацию и сэмплирование на одних весах. Не считайте плавный текст доказательством истинности.


In [ ]:
run_model = False
if run_model:
    import pandas as pd
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_id = "ai-forever/rugpt3small_based_on_gpt2"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id).eval()
    prompts = [
        "В школьном кружке напечатали фигурку. Она",
        "В школьном кружке напечатали модель самолёта. Она",
    ]
    rows = []
    for prompt in prompts:
        encoded = tokenizer(prompt, return_tensors="pt")
        for mode in ("greedy", "sample"):
            torch.manual_seed(7)
            with torch.inference_mode():
                out = model.generate(
                    **encoded, max_new_tokens=30,
                    do_sample=(mode == "sample"),
                    pad_token_id=tokenizer.eos_token_id,
                )
            new = out[0, encoded["input_ids"].shape[1]:]
            rows.append({
                "prompt": prompt, "mode": mode,
                "answer": tokenizer.decode(new, skip_special_tokens=True),
            })
    report = pd.DataFrame(rows)
    print(report.to_string(index=False))


## Разбор результата

Сравните только пары строк, в которых изменилось одно условие. Отметьте обрывы фраз, повторения и хотя бы одно проверяемое фактическое утверждение. Сверьте его с независимым источником и запишите ссылку. Для повторения опыта сохраните идентификатор модели, версии библиотек, prompts, seed и параметры generate.


In [ ]:
if run_model:
    import transformers
    print("torch:", torch.__version__, "transformers:", transformers.__version__)
    print("Модель:", model_id, "новых токенов:", 30, "seed:", 7)


## Самопроверка




### 1. Что получает модель на очередном шаге генерации?

A. Только последний символ.

B. Уже написанную последовательность токенов.

C. Новый набор весов.

D. Только будущий текст.




### 2. Какой токен выбирается жадно из долей 0,60; 0,30; 0,10?

A. Первый с долей 0,60.

B. Второй.

C. Третий.

D. Ни один.




### 3. Что означает max_new_tokens=40?

A. Не более 40 слов во всём тексте.

B. Не более 40 добавленных токенов.

C. Ровно 40 предложений.

D. 40 классов.




### 4. Зачем в примере берётся срез new_ids?

A. Чтобы удалить начальный запрос из печатаемого результата.

B. Чтобы обновить веса.

C. Чтобы перевести текст.

D. Чтобы увеличить длину.




### 5. Что меняет do_sample=True?

A. Включает случайный выбор по распределению.

B. Удаляет модель.

C. Проверяет факты.

D. Обучает сеть.




### 6. Зачем фиксировать seed при опыте со сэмплированием?

A. Для более повторяемого случайного выбора.

B. Чтобы повысить грамотность.

C. Чтобы добавить классы.

D. Чтобы скрыть вход.




### 7. Можно ли считать плавный ответ доказательством истинности даты в нём?

A. Да.

B. Только при жадном выборе.

C. Нет, дату нужно проверить.

D. Только при 40 токенах.




### 8. Что лучше подходит маленькой модели продолжения, а не чат-модели?

A. Начало рассказа.

B. Сложная инструкция с инструментами.

C. Номер файла без текста.

D. Пустой запрос.




### 9. Почему при сравнении двух запусков меняют один параметр за раз?

A. Чтобы связать различие с конкретным условием.

B. Чтобы ускорить интернет.

C. Чтобы заменить данные.

D. Чтобы убрать проверку.




### 10. Какая запись нужна для воспроизведения ответа?

A. Только красивый фрагмент.

B. Модель, её ревизия, вход, версии и настройки.

C. Только seed.

D. Только число слов.




## Подсказки и решения

### Вопрос 1

**Подсказка**


Выбранный токен добавляется ко входу.



**Решение**


**B.** Следующий шаг учитывает имеющееся начало и уже выбранные токены.



### Вопрос 2

**Подсказка**


Жадный выбор берёт максимум.



**Решение**


**A.** Наибольшая доля 0,60 принадлежит первому кандидату.



### Вопрос 3

**Подсказка**


Параметр относится к новому выводу.



**Решение**


**B.** Токен может быть частью слова; исходный prompt не входит в этот лимит.



### Вопрос 4

**Подсказка**


generate возвращает вход и продолжение вместе.



**Решение**


**A.** Срез оставляет только добавленные идентификаторы токенов.



### Вопрос 5

**Подсказка**


Сравните с жадным максимумом.



**Решение**


**A.** Сэмплирование допускает разные продолжения при одном начале.



### Вопрос 6

**Подсказка**


Случайному генератору нужно начальное состояние.



**Решение**


**A.** Seed помогает повторить случайные выборы в той же среде.



### Вопрос 7

**Подсказка**


Модель выбирает продолжение, а не сверяет источник.



**Решение**


**C.** Фактическое утверждение проверяют независимо.



### Вопрос 8

**Подсказка**


Учитывайте задачу, на которой модель обучалась.



**Решение**


**A.** Модель продолжает последовательность токенов.



### Вопрос 9

**Подсказка**


Если изменятся и начало, и длина, причина неясна.



**Решение**


**A.** Контролируемый опыт облегчает интерпретацию результата.



### Вопрос 10

**Подсказка**


На выход влияют веса, токенизация и режим выбора.



**Решение**


**B.** Полная запись позволяет повторить условия генерации.



